# Station Correlation Analysis
This notebook calculates the correlation coefficient between weather stations with at least 100 years of history and that are reporting until the youngest date of the dataset. Because the total number of stations is large, this analysis randomly samples 5 valid stations to produce readable matrices.

It outputs two correlation matrices:
1. Correlation coefficients until the year 2000 (pre-2000).
2. Correlation coefficients from the year 2000 onwards (post-2000).

In [ ]:
import os
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for plots
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (10, 8)

In [ ]:
# Load the dataset from the local data lake
dataset_path = "../data/dwd_historical_lake"

# We only need Station_ID, Date, and Max_Temp_C for the correlation
dataset = ds.dataset(dataset_path, format="parquet")
columns_to_load = ['Station_ID', 'Date', 'Max_Temp_C']

print("Loading dataset into memory...")
df = dataset.to_table(columns=columns_to_load).to_pandas()
df['Date'] = pd.to_datetime(df['Date'])
print(f"Loaded {len(df):,} records.")

In [ ]:
# Group by Station_ID to get min and max dates
station_stats = df.groupby('Station_ID')['Date'].agg(['min', 'max'])
global_max_date = station_stats['max'].max()

# Filter conditions: 
# 1. At least 100 years history -> span >= 100 years (approx 365.25 * 100 days)
# 2. Reporting until the youngest date -> max == global_max_date
valid_stations = station_stats[
    ((station_stats['max'] - station_stats['min']).dt.days >= 365.25 * 100) & 
    (station_stats['max'] == global_max_date)
].index

print(f"Global max date in dataset: {global_max_date.date()}")
print(f"Found {len(valid_stations)} stations meeting the criteria (100+ years & active).")

# Randomly select 5 stations from the valid ones
# Using random_state for reproducibility
selected_stations = pd.Series(valid_stations).sample(n=5, random_state=42).values
print(f"Randomly selected 5 stations for analysis: {selected_stations}")

# Filter main dataframe to only these 5 stations
df_filtered = df[df['Station_ID'].isin(selected_stations)].copy()

In [ ]:
# Pivot the dataframe so that rows are dates and columns are stations
# The values are the daily max temperatures
df_pivot = df_filtered.pivot(index='Date', columns='Station_ID', values='Max_Temp_C')
df_pivot.head()

In [ ]:
# Filter data until the year 2000 (exclusive)
df_pre_2000 = df_pivot[df_pivot.index.year < 2000]

# Calculate Pearson correlation coefficient
corr_pre_2000 = df_pre_2000.corr()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_pre_2000, cmap='coolwarm', vmin=0.5, vmax=1.0, annot=True, fmt=".2f",
            xticklabels=True, yticklabels=True, cbar_kws={'label': 'Correlation Coefficient'})
plt.title("Correlation Matrix of Max Daily Temperatures (Pre-2000)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Filter data from the year 2000 onwards (inclusive)
df_post_2000 = df_pivot[df_pivot.index.year >= 2000]

# Calculate Pearson correlation coefficient
corr_post_2000 = df_post_2000.corr()

# Plot the correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(corr_post_2000, cmap='coolwarm', vmin=0.5, vmax=1.0, annot=True, fmt=".2f",
            xticklabels=True, yticklabels=True, cbar_kws={'label': 'Correlation Coefficient'})
plt.title("Correlation Matrix of Max Daily Temperatures (2000 Onwards)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Optional: Visualize the difference in correlation
corr_diff = corr_post_2000 - corr_pre_2000

plt.figure(figsize=(10, 8))
sns.heatmap(corr_diff, cmap='coolwarm', center=0, annot=True, fmt=".2f",
            xticklabels=True, yticklabels=True, cbar_kws={'label': 'Difference in Correlation'})
plt.title("Difference in Correlation Matrix (Post-2000 minus Pre-2000)", fontsize=16)
plt.tight_layout()
plt.show()